In [ ]:
import sys
import json
import pandas as pd

sys.path.append("..")

from etl_pipeline_local import get_over_15_table

In [ ]:
# Data upload - do not implement
# Bet setup
target_col = "over_15"

# Load data
df_loaded = get_over_15_table()
df_loaded = df_loaded.sort_values('time').reset_index(drop=True)
df = df_loaded.copy()

In [4]:
def get_mask(df, params_dict):
    mask = pd.Series(True, index=df.index)
    features = set()

    for key in params_dict:
        for suffix in ["_cat", "_use_min", "_use_max", "_min", "_max", "_include_missing"]: # ,   "_min_idx", "_max_idx", 
             if key.endswith(suffix):
                features.add(key[:-len(suffix)])
                break

    for feat in features:
        s = df[feat]

        include_missing = params_dict.get(f"{feat}_include_missing", False)
        use_min = params_dict.get(f"{feat}_use_min", True)
        use_max = params_dict.get(f"{feat}_use_max", True)

        feat_mask = pd.Series(True, index=df.index)

        # Categorical filtering
        if f"{feat}_cat" in params_dict:
            feat_mask &= s.isin(params_dict[f"{feat}_cat"])

        # Numerical filtering
        if f"{feat}_min" in params_dict and use_min:
            feat_mask &= s >= params_dict[f"{feat}_min"]

        if f"{feat}_max" in params_dict and use_max:
            feat_mask &= s <= params_dict[f"{feat}_max"]

        if include_missing:
            feat_mask = feat_mask | s.isna()
        else:
            feat_mask = feat_mask & s.notna()

        if params_dict[f"use_{feat}"]:
            mask &= feat_mask

    return mask


def round_to_step(x, step):
    """
    Arrotonda x a un multiplo di 'step'.
    """
    try:
        if not step:
            return str(x)
        elif step <= 0:
            raise ValueError("step deve essere > 0")
        else:
            ratio = x / step
            return round(ratio) * step
    
    except Exception as e: 
        return None

In [ ]:
with open("../../strategies/strategies.json", "r", encoding="utf-8") as f:
    data = json.load(f)

params_dict = data[target_col]['params_dict']
feature_bins_map = data[target_col]['feature_bins_map']

# Feature Engineering
df = df[df["underOver_quote_currentU"] <= 1.7]

df_binned = df.copy()

for feat, step in feature_bins_map.items():
    df[feat] = [round_to_step(x, step) for x in df[feat]]

    if isinstance(step, int):
        df[feat] = df[feat].astype("Int64")


mask = get_mask(df, params_dict)

df_filtered = df[mask]
df_filtered.head()


,time,chance1x2_chance_p1,chance1x2_chance_px,chance1x2_chance_p2,chance1x2_chance_p1x,chance1x2_chance_p2x,chance1x2_chance_p12,chance1x2_chance_pHt1,chance1x2_chance_pHtx,chance1x2_chance_pHt2,chance1x2_chance_p2Ht1x,chance1x2_chance_p2Ht2x,chance1x2_chance_p2Ht12,chance1x2_quote_real1,chance1x2_quote_realx,chance1x2_quote_real2,chance1x2_quote_initial1,chance1x2_quote_initialx,chance1x2_quote_initial2,chance1x2_quote_current1,chance1x2_quote_currentx,chance1x2_quote_current2,chance1x2_quote_diffRealCurr1,chance1x2_quote_diffRealCurrx,chance1x2_quote_diffRealCurr2,chance1x2_quote_diffInitialCurr1,chance1x2_quote_diffInitialCurrx,chance1x2_quote_diffInitialCurr2,chance1x2_bookkeeping_actual,chance1x2_bookkeeping_status,chance1x2_bookkeeping_arrow,chance1x2_bookkeeping_p1,chance1x2_bookkeeping_px,chance1x2_bookkeeping_p2,chance1x2_bookkeeping_avg,chance1x2_comparison_affini,chance1x2_comparison_flashback,chance1x2_flashback_p1,chance1x2_flashback_px,chance1x2_flashback_p2,chance1x2_flashback_pHt1,chance1x2_flashback_pHtx,chance1x2_flashback_pHt2,evaluation_val1x2,evaluation_valUnderOver,evaluation_valMetrica,evaluation_valScala,goalNoGoal_chance_goal,goalNoGoal_chance_noGoal,goalNoGoal_chance_even,goalNoGoal_chance_odd,goalNoGoal_chance_goalHome,goalNoGoal_chance_goalAway,goalNoGoal_multigoal_m13,goalNoGoal_multigoal_m14,goalNoGoal_multigoal_m24,goalNoGoal_multigoal_m35,goalNoGoal_multigoal_m13Home,goalNoGoal_multigoal_m13Away,goalNoGoal_multigoal_m24Home,goalNoGoal_multigoal_m24Away,goalNoGoal_quote_realGG,goalNoGoal_quote_realNG,goalNoGoal_quote_initialGG,goalNoGoal_quote_initialNG,goalNoGoal_quote_currentGG,goalNoGoal_quote_currentNG,goalNoGoal_quote_diffRealCurrGG,goalNoGoal_quote_diffRealCurrNG,goalNoGoal_quote_diffInitialCurrGG,goalNoGoal_quote_diffInitialCurrNG,goalNoGoal_bookkeeping_actual,goalNoGoal_bookkeeping_status,goalNoGoal_bookkeeping_arrow,goalNoGoal_bookkeeping_gg,goalNoGoal_bookkeeping_ng,goalNoGoal_bookkeeping_avg,goalNoGoal_comparison_affini,goalNoGoal_comparison_flashback,goalNoGoal_stats_avgGoalHome,goalNoGoal_stats_avgGoalTakenHome,goalNoGoal_stats_avgGoalAway,goalNoGoal_stats_avgGoalTakenAway,goalNoGoal_flashback_goal,goalNoGoal_flashback_noGoal,goalNoGoal_flashback_m13,goalNoGoal_flashback_m24,goalNoGoal_flashback_m35,team_league,team_home,team_away,underOver_chance_under05HT,underOver_chance_over05HT,underOver_chance_under052HT,underOver_chance_over052HT,underOver_chance_under15HT,underOver_chance_over15HT,underOver_chance_under15,underOver_chance_over15,underOver_chance_under25,underOver_chance_over25,underOver_chance_under35,underOver_chance_over35,underOver_chance_under45,underOver_chance_over45,underOver_quote_realU,underOver_quote_realO,underOver_quote_initialU,underOver_quote_initialO,underOver_quote_currentU,underOver_quote_currentO,underOver_quote_diffRealCurrU,underOver_quote_diffRealCurrO,underOver_quote_diffInitialCurrU,underOver_quote_diffInitialCurrO,underOver_bookkeeping_actual,underOver_bookkeeping_status,underOver_bookkeeping_arrow,underOver_bookkeeping_u,underOver_bookkeeping_o,underOver_bookkeeping_avg,underOver_comparison_affini,underOver_comparison_flashback,underOver_flashback_under05HT,underOver_flashback_over05HT,underOver_flashback_under15,underOver_flashback_over15,underOver_flashback_under25,underOver_flashback_over25,underOver_flashback_under35,underOver_flashback_over35,team_goal,team_goalHt,team_corner,chance1x2_xg_home,chance1x2_xg_away,chance1x2_xg_total,chance1x2_xg,over_15
511,2025-05-25 21:00:00,58.2,22.4,19.4,80.6,41.8,77.6,37.7,42.3,20.0,80.0,62.3,57.7,1.50,5.33,4.36,1.50,3.90,6.50,1.70,3.5,4.75,-11.8,52.3,-8.2,-11.8,11.4,36.8,8.45,3,760603,54.2,26.4,19.4,7.22,1186,220,NaN,NaN,NaN,NaN,NaN,NaN,918493,560003,<NA>,<NA>,51.9,48.1,54.3,45.7,89.0,75.0,65.1,83.6,68.1,49.3,80.1,75.6,54.9,11.1,1.60,2.20,1.83,1.83,2.00,1.72,-20.0,27.9,-8.5,6.4,8.14,0,416049,46.2,53.8,8.32,8939,258,3.0,0.50,0.83,2.33,NaN,NaN,NaN,NaN,NaN,Cile,Universidad Católica,La Serena,29.1,70.9,22.9,77.1,6